# Lecture 16 - Mini-Project: CSV Data Cleaner CLI Tool

## Learning Objectives

- Design and implement a DataCleaner class
- Load CSV data from a file path
- Handle missing values with dropna() and fillna()
- Remove duplicate rows with remove_duplicates()
- Save clean data with save()
- Parse command-line arguments with argparse
- Handle encoding errors and malformed rows gracefully
- Provide progress feedback during processing

## Key Topics

- Designing a DataCleaner class
- Methods: load(), dropna(), fillna(), remove_duplicates(), save()
- Command-line arguments with argparse
- Handling encoding errors and malformed rows
- Progress feedback

## Designing a `DataCleaner` Class

The `DataCleaner` class is the heart of this mini-project. It encapsulates all the logic for loading a CSV file, inspecting it for common data quality issues, cleaning it, and saving the result. The design follows the pattern of sklearn-style transformers: you create an instance, call methods in sequence, and each method returns `self` so calls can be chained.

Key design decisions:
- Store the raw data internally as a list of dictionaries (list of rows).
- Keep track of the original and current row counts so you can report what was removed.
- Each cleaning method prints progress feedback so the user knows what happened.

In [ ]:
import csv

class DataCleaner:
    def __init__(self):
        self.raw_data = None
        self.clean_data = None
        self.original_row_count = 0
        self.current_row_count = 0

    def load(self, filepath, encoding="utf-8"):
        """Load CSV data from a file."""
        self.raw_data = []
        skipped = 0
        try:
            with open(filepath, "r", encoding=encoding) as f:
                reader = csv.DictReader(f)
                if reader.fieldnames is None:
                    raise ValueError("Empty or invalid CSV file")
                self.header = reader.fieldnames
                for row in reader:
                    self.raw_data.append(row)
                self.original_row_count = len(self.raw_data)
                self.clean_data = list(self.raw_data)
                self.current_row_count = self.original_row_count
                print(f"Loaded {self.original_row_count} rows from '{filepath}'")
                print(f"Columns: {', '.join(self.header)}")
        except FileNotFoundError:
            print(f"Error: File '{filepath}' not found.")
            raise
        except UnicodeDecodeError:
            print(f"Unicode error with {encoding}. Trying 'latin-1'...")
            return self.load(filepath, encoding="latin-1")
        return self

    def info(self):
        """Print summary statistics about the current data."""
        print(f"Rows: {self.current_row_count}/{self.original_row_count}")
        print(f"Columns: {self.header}")
        if self.clean_data:
            print(f"Missing values: {sum(1 for r in self.clean_data for v in r.values() if v == '')}")

# Quick test (we'll create a sample CSV first)
print("DataCleaner class designed with chaining in mind.")
print("Continuing to full implementation...")


## Methods: `dropna()`, `fillna()`, `remove_duplicates()`

- **`dropna()`**: Removes rows where **any** field is empty or missing. This is the simplest approach and is appropriate when missing data is rare.
- **`fillna(value)`**: Replaces empty fields with a specified value (e.g., `"Unknown"` for strings or `0` for numbers). This preserves row count.
- **`remove_duplicates()`**: Drops rows that are exact duplicates, keeping the first occurrence. Duplicate rows can skew statistical summaries and model training.

Each method prints a progress message so the user can track what was cleaned and how many rows were affected.

In [ ]:
def dropna(self):
    """Remove rows with any missing (empty) values."""
    before = self.current_row_count
    self.clean_data = [
        row for row in self.clean_data
        if all(v.strip() != "" for v in row.values())
    ]
    self.current_row_count = len(self.clean_data)
    removed = before - self.current_row_count
    print(f"dropna: removed {removed} rows with missing values. {self.current_row_count} remaining.")
    return self

def fillna(self, fill_value="MISSING"):
    """Replace empty values with a specified fill value."""
    count = 0
    for row in self.clean_data:
        for key in row:
            if row[key].strip() == "":
                row[key] = fill_value
                count += 1
    print(f"fillna: filled {count} empty cells with '{fill_value}'.")
    return self

def remove_duplicates(self):
    """Remove duplicate rows (exact match on all fields)."""
    before = self.current_row_count
    seen = set()
    unique = []
    for row in self.clean_data:
        # Create a tuple of values as a hashable key
        key = tuple(row.values())
        if key not in seen:
            seen.add(key)
            unique.append(row)
    self.clean_data = unique
    self.current_row_count = len(self.clean_data)
    removed = before - self.current_row_count
    print(f"remove_duplicates: removed {removed} duplicate rows. {self.current_row_count} remaining.")
    return self

# Bind methods to the class
DataCleaner.dropna = dropna
DataCleaner.fillna = fillna
DataCleaner.remove_duplicates = remove_duplicates

print("Methods dropna(), fillna(), and remove_duplicates() added to DataCleaner.")


## `save()` Method and Full Implementation

The `save()` method writes the cleaned data to a new CSV file. It uses `csv.DictWriter` with the stored header. After saving, it prints a summary of what was accomplished — comparing original vs final row count and listing the operations performed.

This completes the core `DataCleaner` class. Now let's put all the pieces together into a single cohesive implementation.

In [ ]:
def save(self, output_path):
    """Save the cleaned data to a CSV file."""
    with open(output_path, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=self.header)
        writer.writeheader()
        writer.writerows(self.clean_data)
    print(f"Saved {self.current_row_count} rows to '{output_path}'")
    print(f"Data quality summary: removed {self.original_row_count - self.current_row_count} problematic rows "
          f"({((self.original_row_count - self.current_row_count)/max(self.original_row_count,1)*100):.1f}% reduction)")
    return self

DataCleaner.save = save

# Now demonstrate the full implementation
print("DataCleaner is complete. Let's test it on a sample CSV.")


## Command-Line Arguments with `argparse`

The `argparse` module makes it easy to build user-friendly command-line interfaces. For our CSV Cleaner CLI, we'll accept:

- `input`: Path to the input CSV file (positional, required).
- `-o` / `--output`: Output path (default: `cleaned_output.csv`).
- `--dropna` / `--fillna` / `--remove-duplicates`: Flags to enable each cleaning operation.
- `--fill-value`: Value to use for filling missing cells (default: "MISSING").

argparse automatically generates help text, handles validation, and provides useful error messages.

In [ ]:
import argparse

def setup_parser():
    '''Configure the command-line argument parser.'''
    parser = argparse.ArgumentParser(
        description="Clean a CSV file by removing missing values, duplicates, and more.",
        formatter_class=argparse.RawDescriptionHelpFormatter,
        epilog='''Examples:
  python csv_cleaner.py data.csv -o clean.csv --dropna --remove-duplicates
  python csv_cleaner.py data.csv --fillna --fill-value 0
  python csv_cleaner.py data.csv --dropna --fillna --remove-duplicates
'''
    )
    parser.add_argument("input", help="Path to the input CSV file to clean")
    parser.add_argument("-o", "--output", default="cleaned_output.csv",
                        help="Output CSV file path (default: cleaned_output.csv)")
    parser.add_argument("--dropna", action="store_true",
                        help="Remove rows with missing values")
    parser.add_argument("--fillna", action="store_true",
                        help="Fill missing values with --fill-value")
    parser.add_argument("--fill-value", default="MISSING",
                        help="Value to use when filling missing cells (default: MISSING)")
    parser.add_argument("--remove-duplicates", action="store_true",
                        help="Remove duplicate rows")
    return parser

# Demonstrate the parser
parser = setup_parser()
print("argparse configured with the following options:")
parser.print_help()


## Full Pipeline Demonstration

Let's create a realistic (but small) CSV file with common data quality issues — missing values, duplicates, and encoding artifacts — and run the full `DataCleaner` pipeline on it.

We'll then verify the output by loading the cleaned file.

In [ ]:
# Create a sample CSV with quality issues
sample_csv = '''Name,Age,City,Salary
Alice,30,New York,75000
Bob,25,San Francisco,82000
Charlie,,Chicago,68000
Diana,28,Seattle,72000
Alice,30,New York,75000
Eve,35,,
Frank,40,Boston,95000
Diana,28,Seattle,72000
Grace,,,
'''

with open("dirty_data.csv", "w") as f:
    f.write(sample_csv)

print("Created 'dirty_data.csv' with duplicates, missing values, and empty rows.")


In [ ]:
# Run the full cleaning pipeline
cleaner = DataCleaner()
cleaner.load("dirty_data.csv")
cleaner.info()
print("---")
cleaner.dropna()
cleaner.remove_duplicates()
cleaner.save("clean_data.csv")
print("---")

# Verify the cleaned file
print("\nCleaned data contents:")
with open("clean_data.csv", "r") as f:
    print(f.read())


## Handling Encoding Errors and Malformed Rows

Real-world CSV files often have encoding issues (e.g., UTF-8 vs Latin-1) or malformed rows (wrong number of columns). A robust data cleaner must handle these gracefully.

Our `load()` method already tries `utf-8` first, and falls back to `latin-1` on `UnicodeDecodeError`. For malformed rows, we can add a try/except around the CSV parsing. Here's an enhanced version of the loading logic that demonstrates these defensive techniques.

In [ ]:
def safe_load(filepath):
    """Robust CSV loading with encoding fallback and malformed-row handling."""
    data = []
    skipped = 0

    for encoding in ["utf-8", "latin-1", "cp1252"]:
        try:
            with open(filepath, "r", encoding=encoding) as f:
                reader = csv.DictReader(f)
                if reader.fieldnames is None:
                    print("Warning: File appears empty (no header found).")
                    return data
                for i, row in enumerate(reader):
                    try:
                        # Skip rows with wrong number of columns
                        if len(row) != len(reader.fieldnames):
                            skipped += 1
                            continue
                        data.append(row)
                    except Exception as e:
                        skipped += 1
                        print(f"  Skipped row {i+2}: {e}")
            print(f"Loaded with encoding '{encoding}': {len(data)} rows, {skipped} skipped")
            return data
        except UnicodeDecodeError:
            continue
        except Exception as e:
            print(f"Error with '{encoding}': {e}")
            continue

    print("Failed to read file with any encoding.")
    return data

# Test on a file that would normally cause issues
print("safe_load is ready for production use.")
print(f"Test on dirty_data.csv: {len(safe_load('dirty_data.csv'))} rows loaded")


## Data Science Connection

This mini-project integrates everything you've learned in Phase 2: functions, error handling, file I/O, comprehensions, iterators, modules, and OOP. The `DataCleaner` class is a real tool you'll find yourself using in every data science project. It demonstrates how defensive programming — handling encoding issues, malformed rows, and missing values — turns a fragile script into a robust, reusable utility. Adding an `argparse`-based CLI makes it accessible to non-programmers and easily integrable into shell pipelines.